# เครือข่ายประสาทเทียมชนิดวนซ้ำ

ในโมดูลก่อนหน้านี้ เราได้กล่าวถึงการแทนความหมายเชิงสำนวนที่สมบูรณ์ของข้อความ โครงสร้างที่เราใช้จับความหมายรวมของคำในประโยค แต่ไม่ได้พิจารณาถึง **ลำดับ** ของคำ เพราะการดำเนินการรวบรวมที่ตามหลังการฝังคำจะลบข้อมูลนี้ออกไปจากข้อความต้นฉบับ เนื่องจากโมเดลเหล่านี้ไม่สามารถแทนลำดับคำได้ จึงไม่สามารถแก้ปัญหาที่ซับซ้อนหรือคลุมเครือมากขึ้น เช่น การสร้างข้อความหรือการตอบคำถามได้

เพื่อจับความหมายของลำดับข้อความ เราจะใช้สถาปัตยกรรมเครือข่ายประสาทเทียมที่เรียกว่า **เครือข่ายประสาทเทียมชนิดวนซ้ำ** หรือ RNN เมื่อใช้ RNN เราจะส่งประโยคผ่านเครือข่ายทีละโทเค็น และเครือข่ายจะสร้าง **สถานะ** บางอย่างขึ้นมา ซึ่งเราจะส่งสถานะนั้นกลับเข้าเครือข่ายอีกครั้งพร้อมกับโทเค็นถัดไป

![ภาพแสดงตัวอย่างการสร้างเครือข่ายประสาทเทียมชนิดวนซ้ำ](../../../../../translated_images/th/rnn.27f5c29c53d727b5.webp)

เมื่อได้รับลำดับอินพุตของโทเค็น $X_0,\dots,X_n$ RNN จะสร้างลำดับของบล็อกเครือข่ายประสาท และฝึกลำดับนี้แบบ end-to-end ด้วยการย้อนกลับผ่านโพรพาเกชัน แต่ละบล็อกเครือข่ายรับคู่ข้อมูล $(X_i,S_i)$ เป็นอินพุต และให้ค่า $S_{i+1}$ เป็นผลลัพธ์ สถานะสุดท้าย $S_n$ หรือผลลัพธ์ $Y_n$ จะถูกส่งเข้าไปในตัวจำแนกเชิงเส้นเพื่อให้ได้ผลลัพธ์ บล็อกเครือข่ายทั้งหมดใช้ค่าน้ำหนักร่วมกัน และได้รับการฝึกฝน end-to-end โดยผ่านการย้อนกลับแบบครั้งเดียว

> รูปด้านบนแสดงเครือข่ายประสาทเทียมชนิดวนซ้ำในรูปแบบที่คลี่ออก (ทางซ้าย) และในรูปแบบการแทนที่วนซ้ำที่กะทัดรัดกว่า (ทางขวา) สิ่งสำคัญคือต้องตระหนักว่าทุกเซลล์ RNN มี **น้ำหนักที่ใช้ร่วมกัน** เหมือนกัน

เนื่องจากเวกเตอร์สถานะ $S_0,\dots,S_n$ ถูกส่งผ่านเครือข่าย RNN จึงสามารถเรียนรู้การขึ้นต่อกันตามลำดับระหว่างคำ เช่น เมื่อคำ *not* ปรากฏที่ใดที่หนึ่งในลำดับ มันจะเรียนรู้ที่จะปฏิเสธองค์ประกอบบางอย่างภายในเวกเตอร์สถานะ

ภายในแต่ละเซลล์ RNN ประกอบด้วยเมทริกซ์น้ำหนักสองตัว: $W_H$ และ $W_I$ และไบแอส $b$ ในแต่ละขั้นตอนของ RNN เมื่อได้อินพุต $X_i$ และสถานะอินพุต $S_i$ สถานะเอาต์พุตจะถูกคำนวณเป็น $S_{i+1} = f(W_H\times S_i + W_I\times X_i+b)$ โดยที่ $f$ เป็นฟังก์ชันแอกติเวชัน (มักเป็น $\tanh$)

> สำหรับปัญหาอย่างการสร้างข้อความ (ซึ่งเราจะกล่าวถึงในหน่วยถัดไป) หรือการแปลภาษา เราต้องการให้มีค่าเอาต์พุตในแต่ละขั้นตอน RNN ด้วย ในกรณีนี้ มีเมทริกซ์อีกตัวหนึ่ง $W_O$ และเอาต์พุตถูกคำนวณเป็น $Y_i=f(W_O\times S_i+b_O)$

มาดูกันว่าเครือข่ายประสาทเทียมชนิดวนซ้ำช่วยให้เราจำแนกชุดข้อมูลข่าวสารของเราได้อย่างไร

> สำหรับสภาพแวดล้อม sandbox เราจำเป็นต้องรันเซลล์ต่อไปนี้เพื่อให้แน่ใจว่ามีการติดตั้งไลบรารีที่ต้องการและข้อมูลถูกดึงล่วงหน้า หากคุณรันในเครื่องท้องถิ่น คุณสามารถข้ามเซลล์นี้ได้


In [1]:
import sys
!{sys.executable} -m pip install --quiet tensorflow_datasets==4.4.0
!cd ~ && wget -q -O - https://mslearntensorflowlp.blob.core.windows.net/data/tfds-ag-news.tgz | tar xz

In [2]:
import tensorflow as tf
from tensorflow import keras
import tensorflow_datasets as tfds
import numpy as np

# We are going to be training pretty large models. In order not to face errors, we need
# to set tensorflow option to grow GPU memory allocation when required
physical_devices = tf.config.list_physical_devices('GPU') 
if len(physical_devices)>0:
    tf.config.experimental.set_memory_growth(physical_devices[0], True)

ds_train, ds_test = tfds.load('ag_news_subset').values()

เมื่อฝึกอบรมโมเดลขนาดใหญ่ การจัดสรรหน่วยความจำ GPU อาจกลายเป็นปัญหา เรายังอาจจำเป็นต้องทดลองกับขนาดมินิบัตช์ที่แตกต่างกัน เพื่อให้ข้อมูลพอดีกับหน่วยความจำ GPU ของเรา และการฝึกอบรมรวดเร็วเพียงพอ หากคุณกำลังรันโค้ดนี้บนเครื่อง GPU ของคุณเอง คุณอาจทดลองปรับขนาดมินิบัตช์เพื่อเร่งความเร็วการฝึกอบรม

> **หมายเหตุ**: เวอร์ชันของไดรเวอร์ NVidia บางรุ่นเป็นที่ทราบกันว่าไม่ปล่อยหน่วยความจำหลังจากฝึกอบรมโมเดล เรากำลังรันตัวอย่างหลายตัวในสมุดบันทึกนี้ และอาจทำให้หน่วยความจำหมดในบางการตั้งค่า โดยเฉพาะถ้าคุณกำลังทำการทดลองของคุณเองเป็นส่วนหนึ่งของสมุดบันทึกเดียวกัน หากคุณพบข้อผิดพลาดแปลก ๆ เมื่อเริ่มฝึกอบรมโมเดล คุณอาจต้องรีสตาร์ทเคอร์เนลของสมุดบันทึก


In [3]:
batch_size = 16
embed_size = 64

## ตัวจำแนก RNN แบบง่าย

ในกรณีของ RNN แบบง่าย หน่วย recurrent แต่ละตัวเป็นเครือข่ายเชิงเส้นง่าย ๆ ซึ่งรับเวกเตอร์อินพุตและเวกเตอร์สถานะ แล้วสร้างเวกเตอร์สถานะใหม่ ใน Keras สิ่งนี้สามารถแทนได้ด้วยชั้น `SimpleRNN`

แม้ว่าเราจะส่งโทเค็นที่เข้ารหัสแบบ one-hot ไปยังชั้น RNN ได้โดยตรง แต่ก็ไม่ใช่ความคิดที่ดีเนื่องจากมิติที่สูง ดังนั้นเราจะใช้ชั้น embedding เพื่อลดมิติของเวกเตอร์คำ ตามด้วยชั้น RNN และสุดท้ายคือตัวจำแนก `Dense`

> **หมายเหตุ**: ในกรณีที่มิติไม่สูงมาก เช่น เมื่อใช้การแบ่งโทเค็นระดับตัวอักษร อาจจะเหมาะสมที่จะส่งโทเค็นที่เข้ารหัสแบบ one-hot โดยตรงไปยังเซลล์ RNN


In [4]:
vocab_size = 20000

vectorizer = keras.layers.experimental.preprocessing.TextVectorization(
    max_tokens=vocab_size,
    input_shape=(1,))

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.summary()

Model: "sequential"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
text_vectorization (TextVect (None, None)              0         
_________________________________________________________________
embedding (Embedding)        (None, None, 64)          1280000   
_________________________________________________________________
simple_rnn (SimpleRNN)       (None, 16)                1296      
_________________________________________________________________
dense (Dense)                (None, 4)                 68        
Total params: 1,281,364
Trainable params: 1,281,364
Non-trainable params: 0
_________________________________________________________________


> **หมายเหตุ:** เราใช้ชั้นฝังตัวแบบไม่ผ่านการฝึกอบรมที่นี่เพื่อความเรียบง่าย แต่เพื่อผลลัพธ์ที่ดีกว่า เราสามารถใช้ชั้นฝังตัวที่ผ่านการฝึกอบรมล่วงหน้าด้วย Word2Vec ตามที่อธิบายไว้ในหน่วยก่อนหน้า จะเป็นการฝึกที่ดีสำหรับคุณในการปรับโค้ดนี้ให้ทำงานกับฝังตัวที่ผ่านการฝึกอบรมล่วงหน้า

ตอนนี้เรามาฝึก RNN ของเรากัน RNN โดยทั่วไปค่อนข้างยากที่จะฝึก เพราะเมื่อเซลล์ RNN ถูกขยายตามความยาวลำดับ จำนวนชั้นที่เกี่ยวข้องกับการย้อนกลับผ่านจึงมีจำนวนมาก ดังนั้นเราจึงต้องเลือกอัตราการเรียนรู้ที่น้อยลง และฝึกเครือข่ายบนชุดข้อมูลที่ใหญ่ขึ้นเพื่อให้ได้ผลลัพธ์ที่ดี ซึ่งอาจใช้เวลาค่อนข้างนาน ดังนั้นการใช้ GPU จึงเป็นที่นิยม

เพื่อเร่งความเร็ว เราจะฝึกโมเดล RNN เฉพาะกับหัวข้อข่าวโดยไม่ใช้คำอธิบาย คุณสามารถลองฝึกกับคำอธิบายและดูว่าคุณจะทำให้โมเดลฝึกได้นะ


In [5]:
def extract_title(x):
    return x['title']

def tupelize_title(x):
    return (extract_title(x),x['label'])

print('Training vectorizer')
vectorizer.adapt(ds_train.take(2000).map(extract_title))

Training vectorizer


In [6]:
model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize_title).batch(batch_size),validation_data=ds_test.map(tupelize_title).batch(batch_size))

7500/7500 [==============================] - 82s 11ms/step - loss: 0.6629 - acc: 0.7623 - val_loss: 0.5559 - val_acc: 0.7995


> **หมายเหตุ** ว่าความแม่นยำอาจต่ำกว่าในนี้ เนื่องจากเรากำลังฝึกอบรมเฉพาะบนหัวข้อข่าวเท่านั้น


## Revisiting variable sequences 

Remember that the `TextVectorization` layer will automatically pad sequences of variable length in a minibatch with pad tokens. It turns out that those tokens also take part in training, and they can complicate convergence of the model.

There are several approaches we can take to minimize the amount of padding. One of them is to reorder the dataset by sequence length and group all sequences by size. This can be done using the `tf.data.experimental.bucket_by_sequence_length` function (see [documentation](https://www.tensorflow.org/api_docs/python/tf/data/experimental/bucket_by_sequence_length)). 

Another approach is to use **masking**. In Keras, some layers support additional input that shows which tokens should be taken into account when training. To incorporate masking into our model, we can either include a separate `Masking` layer ([docs](https://keras.io/api/layers/core_layers/masking/)), or we can specify the `mask_zero=True` parameter of our `Embedding` layer.

> **Note**: This training will take around 5 minutes to complete one epoch on the whole dataset. Feel free to interrupt training at any time if you run out of patience. What you can also do is limit the amount of data used for training, by adding `.take(...)` clause after `ds_train` and `ds_test` datasets.


In [7]:
def extract_text(x):
    return x['title']+' '+x['description']

def tupelize(x):
    return (extract_text(x),x['label'])

model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size,embed_size,mask_zero=True),
    keras.layers.SimpleRNN(16),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),validation_data=ds_test.map(tupelize).batch(batch_size))

7500/7500 [==============================] - 371s 49ms/step - loss: 0.5401 - acc: 0.8079 - val_loss: 0.3780 - val_acc: 0.8822


ตอนนี้ที่เราใช้การปิดบัง เราสามารถฝึกโมเดลได้บนชุดข้อมูลทั้งหมดของหัวข้อและคำอธิบาย

> **หมายเหตุ**: คุณสังเกตไหมว่าเรากำลังใช้เวกเตอร์ไลเซอร์ที่ฝึกบนหัวข้อข่าว ไม่ใช่ทั้งเนื้อหาบทความ? ซึ่งอาจทำให้โทเค็นบางตัวถูกละเลย ดังนั้นจึงควรฝึกเวกเตอร์ไลเซอร์ใหม่ อย่างไรก็ตาม มันอาจมีผลกระทบน้อยมาก ดังนั้นเราจะใช้เวกเตอร์ไลเซอร์ที่ฝึกไว้ก่อนหน้าเพื่อความเรียบง่าย


## LSTM: หน่วยความจำระยะสั้นยาว

หนึ่งในปัญหาหลักของ RNN คือ **กราดิเอนต์จางหาย** RNN สามารถมีความยาวมาก และอาจประสบปัญหาในการส่งผ่านกราดิเอนต์กลับไปยังเลเยอร์แรกของเครือข่ายในระหว่างการย้อนกลับเมื่อฝึก (backpropagation) เมื่อเกิดเหตุการณ์นี้ เครือข่ายจะไม่สามารถเรียนรู้ความสัมพันธ์ระหว่างโทเคนที่อยู่ห่างไกลกันได้ วิธีหนึ่งในการหลีกเลี่ยงปัญหานี้คือการเพิ่ม **การจัดการสถานะโดยชัดเจน** ผ่านการใช้ **เกต** สองสถาปัตยกรรมที่ใช้เกตกันบ่อยที่สุดคือ **หน่วยความจำระยะสั้นยาว** (LSTM) และ **หน่วยสลับผ่านแบบเกต** (GRU) เราจะพูดถึง LSTM ที่นี่

![รูปภาพแสดงตัวอย่างเซลล์หน่วยความจำระยะสั้นยาว](../../../../../lessons/5-NLP/16-RNN/images/long-short-term-memory-cell.svg)

เครือข่าย LSTM ถูกจัดวางในลักษณะคล้ายกับ RNN แต่มีสถานะสองตัวที่ถูกส่งต่อจากเลเยอร์หนึ่งไปยังอีกเลเยอร์: สถานะจริง $c$ และเวคเตอร์ซ่อน $h$ ที่หน่วยแต่ละตัว เวคเตอร์ซ่อน $h_{t-1}$ จะถูกผสมกับอินพุต $x_t$ และร่วมกันควบคุมสิ่งที่จะเกิดขึ้นกับสถานะ $c_t$ และผลลัพธ์ $h_t$ ผ่าน **เกต** แต่ละเกตมีฟังก์ชัน sigmoid (ผลลัพธ์อยู่ในช่วง $[0,1]$) ซึ่งสามารถคิดได้ว่าเป็นหน้ากากบิตที่ถูกคูณกับเวคเตอร์สถานะ LSTM มีเกตดังนี้ (จากซ้ายไปขวาในภาพด้านบน):
* **เกตลืม** ที่กำหนดว่าส่วนประกอบใดของเวคเตอร์ $c_{t-1}$ ที่เราต้องลืม และส่วนประกอบใดที่ต้องส่งผ่าน
* **เกตอินพุต** ที่กำหนดว่าข้อมูลจากเวคเตอร์อินพุตและเวคเตอร์ซ่อนก่อนหน้าจะถูกรวมเข้ากับเวคเตอร์สถานะมากน้อยเพียงใด
* **เกตเอาต์พุต** ที่ใช้เวคเตอร์สถานะใหม่และตัดสินใจว่าส่วนประกอบใดของเวคเตอร์นั้นจะถูกใช้ในการสร้างเวคเตอร์ซ่อนใหม่ $h_t$

ส่วนประกอบของสถานะ $c$ สามารถคิดได้ว่าเป็นธงที่สามารถเปิดหรือปิดได้ เช่น เมื่อเราพบชื่อ *Alice* ในลำดับ เราจะเดาว่าหมายถึงผู้หญิง และยกธงในสถานะที่บ่งชี้ว่าเรามีนามที่เป็นเพศหญิงในประโยค เมื่อเราพบคำว่า *and Tom* เราจะยกธงที่บ่งชี้ว่านามเป็นพหูพจน์ ดังนั้นโดยการจัดการสถานะ เราสามารถติดตามคุณสมบัติไวยากรณ์ของประโยคได้

> **หมายเหตุ**: นี่เป็นแหล่งข้อมูลที่ดีมากสำหรับการเข้าใจโครงสร้างภายในของ LSTM: [Understanding LSTM Networks](https://colah.github.io/posts/2015-08-Understanding-LSTMs/) โดย Christopher Olah

แม้โครงสร้างภายในของเซลล์ LSTM จะดูซับซ้อน Keras ซ่อนการทำงานนี้ไว้ในเลเยอร์ `LSTM` ดังนั้นสิ่งเดียวที่เราต้องทำในตัวอย่างข้างต้นคือเปลี่ยนเลเยอร์แบบ recurrent:


In [8]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, embed_size),
    keras.layers.LSTM(8),
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(8),validation_data=ds_test.map(tupelize).batch(8))

15000/15000 [==============================] - 188s 13ms/step - loss: 0.5692 - acc: 0.7916 - val_loss: 0.3441 - val_acc: 0.8870


> **หมายเหตุ** ว่าการฝึก LSTMs ก็ช้ามากเช่นกัน และคุณอาจไม่เห็นการเพิ่มขึ้นของความแม่นยำมากนักในช่วงเริ่มต้นของการฝึก คุณอาจจำเป็นต้องฝึกต่อไปสักระยะเพื่อให้ได้ความแม่นยำที่ดี


## RNN แบบสองทิศทางและหลายชั้น

ในตัวอย่างของเราจนถึงตอนนี้ เครือข่าย recurring ทำงานตั้งแต่ต้นลำดับจนถึงปลายลำดับ สิ่งนี้รู้สึกเป็นธรรมชาติสำหรับเราเพราะมันตามทิศทางเดียวกับที่เราอ่านหรือฟังเสียงพูด อย่างไรก็ตาม สำหรับสถานการณ์ที่ต้องการเข้าถึงลำดับข้อมูลแบบสุ่ม การคำนวณ recurrent ทั้งสองทิศทางจะสมเหตุสมผลกว่า RNN ที่อนุญาตให้คำนวณทั้งสองทิศทางเรียกว่า RNN **แบบสองทิศทาง** และสามารถสร้างได้โดยการครอบชั้น recurrent ด้วยชั้น `Bidirectonal` พิเศษ

> **หมายเหตุ**: ชั้น `Bidirectional` สร้างสำเนาชั้นภายในสองชุด และตั้งสมบัติ `go_backwards` ของสำเนาชุดหนึ่งเป็น `True` เพื่อให้ทำงานในทิศทางตรงกันข้ามของลำดับ

เครือข่าย recurrent ไม่ว่าจะเป็นแบบทิศทางเดียวหรือสองทิศทาง สามารถจับรูปแบบภายในลำดับ และเก็บไว้ในเวกเตอร์สถานะหรือส่งกลับเป็นผลลัพธ์ เหมือนกับเครือข่าย convolutional เราสามารถสร้างชั้น recurrent เพิ่มเติมต่อจากชั้นแรกเพื่อจับรูปแบบระดับสูงขึ้น ซึ่งสร้างจากรูปแบบระดับต่ำที่ชั้นแรกดึงออกมา สิ่งนี้นำไปสู่แนวคิดของ **RNN หลายชั้น** ซึ่งประกอบด้วยเครือข่าย recurrent สองชั้นขึ้นไป โดยที่ผลลัพธ์ของชั้นก่อนหน้าถูกส่งต่อไปยังชั้นถัดไปเป็นอินพุต

![ภาพแสดง Multilayer long-short-term-memory- RNN](../../../../../translated_images/th/multi-layer-lstm.dd975e29bb2a59fe.webp)

*ภาพจาก [โพสต์ที่ยอดเยี่ยมนี้](https://towardsdatascience.com/from-a-lstm-cell-to-a-multilayer-lstm-network-with-pytorch-2899eb5696f3) โดย Fernando López*

Keras ทำให้การสร้างเครือข่ายเหล่านี้เป็นเรื่องง่าย เพราะคุณแค่เพิ่มชั้น recurrent เข้าไปในโมเดล สำหรับชั้นทั้งหมดยกเว้นชั้นสุดท้าย เราต้องระบุพารามิเตอร์ `return_sequences=True` เพราะเราต้องการให้ชั้นส่งกลับสถานะระหว่างทั้งหมด ไม่ใช่แค่สถานะสุดท้ายของการคำนวณ recurrent

มาสร้าง LSTM แบบสองชั้นสองทิศทางสำหรับปัญหาการจัดหมวดหมู่ของเรากันเถอะ

> **หมายเหตุ** โค้ดนี้ใช้เวลานานมากอีกครั้งในการประมวลผล แต่ให้ความแม่นยำสูงสุดที่เราเห็นจนถึงตอนนี้ ดังนั้นอาจคุ้มค่าที่จะรอและดูผลลัพธ์


In [9]:
model = keras.models.Sequential([
    vectorizer,
    keras.layers.Embedding(vocab_size, 128, mask_zero=True),
    keras.layers.Bidirectional(keras.layers.LSTM(64,return_sequences=True)),
    keras.layers.Bidirectional(keras.layers.LSTM(64)),    
    keras.layers.Dense(4,activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy',metrics=['acc'], optimizer='adam')
model.fit(ds_train.map(tupelize).batch(batch_size),
          validation_data=ds_test.map(tupelize).batch(batch_size))

5045/7500 [===================>..........] - ETA: 2:33 - loss: 0.3709 - acc: 0.8706

## RNNs สำหรับงานอื่นๆ

จนถึงตอนนี้ เราได้มุ่งเน้นไปที่การใช้ RNNs เพื่อจำแนกลำดับของข้อความ แต่พวกมันสามารถจัดการกับงานอื่นๆ ได้มากขึ้น เช่น การสร้างข้อความและการแปลภาษาแบบเครื่อง — เราจะพิจารณางานเหล่านั้นในหน่วยถัดไป


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ปฏิเสธความรับผิดชอบ**:
เอกสารนี้ได้รับการแปลโดยใช้บริการแปลภาษา AI [Co-op Translator](https://github.com/Azure/co-op-translator) ขณะที่เราพยายามให้ความถูกต้อง โปรดทราบว่าการแปลโดยอัตโนมัติอาจมีข้อผิดพลาดหรือความไม่ถูกต้อง เอกสารต้นฉบับในภาษาต้นทางควรถูกพิจารณาเป็นแหล่งข้อมูลที่เชื่อถือได้ สำหรับข้อมูลที่สำคัญ แนะนำให้ใช้การแปลโดยมนุษย์มืออาชีพ เราไม่รับผิดชอบต่อความเข้าใจผิดหรือการตีความที่ผิดพลาดที่เกิดขึ้นจากการใช้การแปลนี้
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
